<a href="https://colab.research.google.com/github/9terry-student/Data_Structure_and_algorhithm/blob/main/numpy_RWKV_no_hints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np

def Time_Mixing(x,x_prev,num,den):
  W_r=np.random.randn(d_model,d_model)
  u_r=1/(1+np.exp(-1*np.random.randn(d_model,)))
  R=W_r@(u_r*x+(1-u_r)*x_prev)

  W_k=np.random.randn(d_model,d_model)
  u_k=1/(1+np.exp(-1*np.random.randn(d_model,)))
  K=W_k@(u_k*x+(1-u_k)*x_prev)

  W_v=np.random.randn(d_model,d_model)
  u_v=1/(1+np.exp(-1*np.random.randn(d_model,)))
  V=W_v@(u_v*x+(1-u_v)*x_prev)

  output=(np.exp(u_k+K)*V+num)/(np.exp(u_k+K)+den)
  output*=1/(1+np.exp(-1*R))

  return output,num,den

def Channel_Mixing(x,x_prev):
  W_r=np.random.randn(d_model,d_model)
  u_r=1/(1+np.exp(-1*np.random.randn(d_model,)))
  W_k=np.random.randn(d_model,d_model)
  u_k=1/(1+np.exp(-1*np.random.randn(d_model,)))

  R=W_r@(u_r*x+(1-u_r)*x_prev)
  K=W_k@(u_k*x+(1-u_k)*x_prev)

  output=((np.maximum(0,K))**2)/(1+np.exp(-1*R))
  return output

def layernorm(x):
  output=(x-np.mean(x,axis=-1,keepdims=True))/(np.std(x,axis=-1,keepdims=True)+10**(-9))
  return output

def RWKV(x):
  seq_len,d_model=x.shape
  x_prev=np.zeros(d_model)
  num=np.zeros(d_model)
  den=np.zeros(d_model)
  outputs=[]
  for i in range(seq_len):
    out,num,den=Time_Mixing(layernorm(x[i]),x_prev,num,den)
    out+=x[i]

    out+=Channel_Mixing(layernorm(out),x_prev)
    outputs.append(out)
    x_prev=x[i]
  output=np.array(outputs)
  return output

In [5]:
# test
np.random.seed(42)
seq_len = 4
d_model = 16

x = np.random.randn(seq_len, d_model)
output = RWKV(x)
print("output shape:", output.shape)  # (4, 16)


output shape: (4, 16)
